# Pattern ① LLMモデル連携（OpenAI互換エンドポイント）

## 概要

DatabricksのFoundation Model API（FMAPI）はOpenAI互換APIを提供しています。  
DifyのOpenAI-API-compatibleプラグインを使うことで、Databricksで管理するLLMをDifyから直接利用できます。

## 使用モデル: `databricks-gpt-5-2`

GPT系モデルはDifyプラグインが送信する `user` パラメータをそのまま受け入れるため、  
**プラグインの修正なしで接続可能**です。

> ⚠️ Claude系・Gemini系モデルは `user` パラメータを拒否するため、プラグイン側のパッチが必要になります。

## アーキテクチャ

```
Dify App ──OpenAI互換API──▶ Databricks FMAPI
                               │
                          databricks-gpt-5-2
                        (Foundation Model API)
```

In [ ]:
%run ./config

## Step 1: Databricks側の確認

In [ ]:
import os
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

print("=" * 60)
print("Dify設定用の情報（コピーしてDifyに貼り付け）")
print("=" * 60)
print(f"\nModel Name:     {NOCODE_LLM_ENDPOINT_NAME}")
print(f"API endpoint URL:  https://{host}/serving-endpoints/{NOCODE_LLM_ENDPOINT_NAME}/invocations")
print(f"API Key:   {token[:10]}... (PATトークンを使用)")
print(f"Completion mode: Chat")

## Step 2: Dify側の設定

### 2.1 Model Providerの追加

1. Difyの **Settings** → **Model Provider** を開く
2. **OpenAI-API-compatible** を選択（事前にプラグインをインストール済みであること）
3. 以下を入力:
   - **Model Name**: `databricks-gpt-5-2`
   - **Model Type**: `LLM`
   - **API endpoint URL**: `https://<workspace_url>/serving-endpoints/databricks-gpt-5-2/invocations`
   - **API Key**: DatabricksのPATトークン
   - **Completion mode**: `Chat`
4. **Save** をクリック

> API endpoint URLは上のStep 1のセルで表示された値をそのままコピーしてください。

### 2.2 チャットアプリの作成

1. **Studio** → **Create from Blank** → **Chatbot**
2. オーケストレーション画面でモデルに **`databricks-gpt-5-2`** を選択
3. System Promptに以下を入力:
   ```
   あなたはTechNovaのカスタマーサポートです。丁寧に日本語で回答してください。
   ```
4. **Publish** → 右側のテスト画面で動作確認

## Step 3: Python側からの動作確認（比較用）

In [ ]:
import requests
import json

url = f"https://{host}/serving-endpoints/{NOCODE_LLM_ENDPOINT_NAME}/invocations"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}
data = {
    "messages": [
        {"role": "system", "content": "あなたはTechNovaのカスタマーサポートです。"},
        {"role": "user", "content": "SmartWatch X100について教えてください"}
    ],
    "max_tokens": 500
}

response = requests.post(url, headers=headers, json=data)
result = response.json()
print("Databricks API応答:")
print(result.get("choices", [{}])[0].get("message", {}).get("content", "エラー"))

## メリット

| 観点 | Dify単体 | Databricks連携 |
|------|---------|--------------|
| APIキー管理 | 各Difyアプリで個別管理 | Databricks PATで一元管理 |
| 利用量追跡 | 困難 | AI Gatewayで自動追跡 |
| コスト管理 | 各アプリ個別 | ダッシュボードで一元可視化 |
| レート制限 | なし | AI Gatewayで設定可能 |
| モデル切替 | 各アプリで変更 | エンドポイント設定で一括変更 |

## 補足: `user` パラメータの互換性

DifyのOpenAI-API-compatibleプラグインは、OpenAI仕様に従い `user` パラメータをリクエストに含めます。

| モデル | `user`パラメータ | Difyとの相性 |
|--------|-----------------|-------------|
| **GPT系**（gpt-5.2等） | ✅ 受け入れる | **そのまま動作** |
| Claude系 | ❌ 拒否する | プラグインのパッチが必要 |
| Gemini系 | ❌ 拒否する | プラグインのパッチが必要 |
| AI Gateway（カスタムルート） | ❌ 拒否する | プラグインのパッチが必要 |

Claude/Gemini系を使いたい場合は、プラグインの `models/llm/llm.py` で `user = None` を設定するパッチが必要です。